# Stage-B Dynamic Ensemble v5 — Competence in **Decision Space**

**Extends `two_stage_stageB_dynamic_v4.ipynb`** (v4-hybrid 0.6295, v4-dynamic 0.6262, v3 static 0.6334).

## The gap this closes

v4 estimated per-instance competence by kNN in **RobustScaler feature space** (44 dims). That is
**not** the standard DES formulation. Woloszynski & Kurzynski, KNORA and META-DES all define the
competence neighbourhood in **decision space** — the region is characterised by the pool's own
*output profiles* $\tilde x = [P_1(\cdot\mid x), \dots, P_M(\cdot\mid x)] \in \mathbb{R}^{M \times 9}$,
not by raw features.

This matters because it attacks v4's own diagnosis head-on. v4 concluded:

> *"neighbourhoods in the 44-feature space are too impure to say which model to trust locally"*

which is precisely the argument for **not defining neighbourhoods in that space**. Two flows can sit
far apart in feature space while the pool treats them identically, and two near-neighbours in feature
space can straddle a decision boundary where members disagree sharply. Decision space groups rows by
*how the pool behaves*, which is what competence is actually about.

Everything else is held fixed — same prune, splits, seeds, 7-member pool, same combiners, same
protocol. **Only the metric space for the kNN changes.** So any difference is attributable to that
one choice.

## The diagnostic that decides it (Section 4)

Before any sweep, this notebook measures whether the competence estimator carries **any signal at
all**, in either space:

$$\text{AUC}_m \;=\; \text{ROC-AUC}\big(\hat w_m(x) \;\to\; \mathbb{1}[\text{member } m \text{ correct on } x]\big)$$

If $\text{AUC}_m \approx 0.5$, the estimated competence is uninformative about actual correctness and
**no combiner built on it can work** — that would explain every dynamic result in this project at a
stroke, and it is a far more fundamental finding than another score comparison. If decision space
lifts the AUC materially above feature space, the v4 negative result was a space problem, not a
dynamic-weighting problem.

> **Runtime ~5 min.** Reuses `cache_oof_v4.npz`; the only expensive step is refitting the pool for the
> test pass (~60 s).

## 1. Setup - identical to v4, loading the cached OOF

In [1]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)

attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

y_arr   = y_trf.values
att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values

SPECS = ([ (f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0) ]
         + [("XGB smote2000", ("smote", 2000)),
            ("RF a=0.25",     ("rf", 0.25)),
            ("ET a=0.25",     ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)

CACHE = Path("cache_oof_v4.npz")
assert CACHE.exists(), "run two_stage_stageB_dynamic_v4.ipynb first to build the OOF cache"
z = np.load(CACHE)
OOF = [z[f"m{m}"] for m in range(M)]
oof_gate = z["gate"]
print("loaded cached OOF | features %d | train+val %d (%d attack) | test %d"
      % (X.shape[1], len(y_trf), att_trf.sum(), len(y_test)))

loaded cached OOF | features 44 | train+val 244938 (64936 attack) | test 61235


In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

def sample_w(y_arr_, wc):
    w = wc[y_arr_].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    else: raise ValueError(kind)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g = make_xgb(); g.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g

## 2. Two competence spaces over the same reference set

`FEAT` - RobustScaler over the 44 pruned features. This is exactly what v4 used.

`DEC` - the pool's **output profile**: the M x 9 = 63 stacked probabilities, flattened. Probabilities
are already on a common [0,1] scale, so no rescaling is applied (rescaling would distort the simplex
geometry that makes this space meaningful).

Both index the *same* 32,468 COMB attack rows, so the only difference is the distance metric.

In [3]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr)), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr)
ref_idx = comb_idx[att_trf[comb_idx]]
y_ref   = fam_trf[ref_idx]
KMAX    = 100

def profile(probas_list):
    return np.hstack(probas_list).astype(np.float32)

# --- FEATURE space (v4's choice) ---
scaler_F = RobustScaler().fit(X_trf.iloc[ref_idx])
NN_F = NearestNeighbors(n_neighbors=KMAX, algorithm="brute", n_jobs=-1).fit(
    scaler_F.transform(X_trf.iloc[ref_idx]))

# --- DECISION space (standard DES formulation) ---
REF_PROF = profile([OOF[m][ref_idx] for m in range(M)])
NN_D = NearestNeighbors(n_neighbors=KMAX, algorithm="brute", n_jobs=-1).fit(REF_PROF)

CORRECT = np.stack([(OOF[m][ref_idx].argmax(1) == y_ref) for m in range(M)])
WC_REF  = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)

print("reference set: %d COMB attack rows" % len(ref_idx))
print("  FEAT space: %d dims | DEC space: %d dims" % (X_trf.shape[1], REF_PROF.shape[1]))

reference set: 32468 COMB attack rows
  FEAT space: 44 dims | DEC space: 63 dims


In [4]:
def comp_ola(dist, idx, k, class_weighted=False, dist_weighted=False):
    nb = idx[:, :k]
    wn = WC_REF[y_ref[nb]] if class_weighted else np.ones(nb.shape)
    if dist_weighted:
        wn = wn / (dist[:, :k] + 1e-6)
    denom = wn.sum(1)
    W = np.empty((nb.shape[0], M))
    for m in range(M):
        W[:, m] = (CORRECT[m][nb] * wn).sum(1) / denom
    return W

def comp_lca(probas, idx, k):
    nb = idx[:, :k]
    yn = y_ref[nb]
    W  = np.empty((nb.shape[0], M))
    for m in range(M):
        pq   = probas[m].argmax(1)
        mask = yn == pq[:, None]
        cnt  = mask.sum(1)
        W[:, m] = np.where(cnt > 0,
                           (CORRECT[m][nb] & mask).sum(1) / np.maximum(cnt, 1),
                           CORRECT[m][nb].mean(1))
    return W

def fuse_dyn(probas, Wdyn, p=1.0, Wclass=None):
    if p == 0:
        Wp = np.ones_like(Wdyn) / Wdyn.shape[1]
    else:
        Wp = np.power(np.clip(Wdyn, 1e-6, None), p)
        Wp = Wp / Wp.sum(1, keepdims=True)
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        coef = Wp[:, m:m + 1] if Wclass is None else Wp[:, m:m + 1] * Wclass[m][None, :]
        S += coef * probas[m]
    return S.argmax(1)

def fuse_static(probas, Wclass):
    S = np.zeros(probas[0].shape, np.float64)
    for m in range(M):
        S += probas[m] * Wclass[m][None, :]
    return S.argmax(1)

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)

def fit_W(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for g in GRID_W:
                    W[m_, c] = g
                    if W[:, c].sum() == 0:
                        continue
                    s = f1_score(ytrue, fuse_static(probas, W), average="macro", zero_division=0)
                    if s > best + 1e-7:
                        best, cur = s, g
                W[m_, c] = cur
    return W, best

t0 = time.time()
W_class, s_comb = fit_W([OOF[m][ref_idx] for m in range(M)], y_ref)
print("W fitted on %d COMB attack rows in %.0fs -> COMB Stage-B mF1 %.4f"
      % (len(ref_idx), time.time() - t0, s_comb))

W fitted on 32468 COMB attack rows in 11s -> COMB Stage-B mF1 0.6183


## 3. Neighbourhoods for the SELV gate-positive rows, in both spaces

In [5]:
gmask_selv = oof_gate[selv_idx] == 1
gpos_selv  = selv_idx[gmask_selv]
P_selv     = [OOF[m][gpos_selv] for m in range(M)]

t0 = time.time()
dF_selv, iF_selv = NN_F.kneighbors(scaler_F.transform(X_trf.iloc[gpos_selv]))
dD_selv, iD_selv = NN_D.kneighbors(profile(P_selv))
print("neighbourhoods for %d gate-positive SELV rows in %.0fs (both spaces)"
      % (len(gpos_selv), time.time() - t0))

# neighbourhood purity: fraction of the k nearest reference rows sharing the query's true family
y_selv_true = fam_trf[gpos_selv]
att_only    = y_selv_true >= 0
for tag, idx in (("FEAT", iF_selv), ("DEC", iD_selv)):
    for k in (10, 30, 100):
        pur = (y_ref[idx[att_only, :k]] == y_selv_true[att_only, None]).mean()
        print("  %-4s k=%-3d neighbourhood purity %.4f" % (tag, k, pur))

neighbourhoods for 35153 gate-positive SELV rows in 4s (both spaces)
  FEAT k=10  neighbourhood purity 0.5721
  FEAT k=30  neighbourhood purity 0.5465
  FEAT k=100 neighbourhood purity 0.5147
  DEC  k=10  neighbourhood purity 0.6698
  DEC  k=30  neighbourhood purity 0.6686
  DEC  k=100 neighbourhood purity 0.6666


## 4. Does the competence estimate carry any signal? — the decisive diagnostic

For each member, treat the estimated competence $\hat w_m(x)$ as a score for predicting whether that
member is **actually correct** on $x$, and compute ROC-AUC over SELV attack rows.

- **AUC ~ 0.50** - the estimate is noise. No combiner built on it can work, in any space. This would
  explain DES v1, v4 and everything in between with a single measurement.
- **AUC materially > 0.50** - there is signal; the question becomes whether it survives the argmax.

This is run *before* the sweep so the conclusion is not reverse-engineered from scores.

In [6]:
Wd_F = comp_ola(dF_selv, iF_selv, 100, class_weighted=True)
Wd_D = comp_ola(dD_selv, iD_selv, 100, class_weighted=True)

rows = []
for m in range(M):
    actual = (P_selv[m][att_only].argmax(1) == y_selv_true[att_only]).astype(int)
    r = {"member": MODELS[m], "actual acc": round(actual.mean(), 4)}
    for tag, Wd in (("FEAT", Wd_F), ("DEC", Wd_D)):
        s = Wd[att_only, m]
        r[tag + " AUC"] = round(roc_auc_score(actual, s), 4) if actual.min() != actual.max() else np.nan
    rows.append(r)
auc_tab = pd.DataFrame(rows)
auc_tab["lift"] = (auc_tab["DEC AUC"] - auc_tab["FEAT AUC"]).round(4)
print(auc_tab.to_string(index=False))
print("\nmean AUC  FEAT %.4f | DEC %.4f | lift %+.4f"
      % (auc_tab["FEAT AUC"].mean(), auc_tab["DEC AUC"].mean(),
         auc_tab["DEC AUC"].mean() - auc_tab["FEAT AUC"].mean()))
print("AUC 0.50 = the competence estimate is pure noise")

for tag, Wd in (("FEAT", Wd_F), ("DEC", Wd_D)):
    Wn = Wd / Wd.sum(1, keepdims=True)
    print("%-4s mean |deviation from uniform| %.4f  (uniform %.4f)"
          % (tag, np.abs(Wn - 1 / M).mean(), 1 / M))

       member  actual acc  FEAT AUC  DEC AUC   lift
    XGB a=0.0      0.7413    0.8041   0.8381 0.0340
   XGB a=0.25      0.7409    0.8077   0.8392 0.0315
    XGB a=0.5      0.7345    0.8145   0.8449 0.0304
    XGB a=1.0      0.6928    0.8406   0.8824 0.0418
XGB smote2000      0.7417    0.8054   0.8384 0.0330
    RF a=0.25      0.7209    0.8316   0.8656 0.0340
    ET a=0.25      0.7159    0.8311   0.8684 0.0373

mean AUC  FEAT 0.8193 | DEC 0.8539 | lift +0.0346
AUC 0.50 = the competence estimate is pure noise
FEAT mean |deviation from uniform| 0.0064  (uniform 0.1429)
DEC  mean |deviation from uniform| 0.0060  (uniform 0.1429)


## 5. Variant sweep on SELV — both spaces, identical combiners

In [7]:
def score10_g(fam_gpos, ye_arr, gmask, tag):
    yp = np.zeros(len(ye_arr), int)
    if len(fam_gpos):
        yp[gmask] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_gpos)
    return dict(variant=tag,
                e2e_mF1=round(f1_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mP =round(precision_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_mR =round(recall_score(ye_arr, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye_arr).mean()), 4))

def run_all(P_g, spaces, ye_arr, gmask):
    out = [score10_g(fuse_static(P_g, W_class), ye_arr, gmask, "STATIC v3: per-class W"),
           score10_g(fuse_static(P_g, np.ones((M, NC)) / M), ye_arr, gmask, "STATIC: equal average")]
    for tag, (dist, idx) in spaces.items():
        for k in (30, 100):
            out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, k, class_weighted=True)),
                                 ye_arr, gmask, f"[{tag}] DYN OLA-cw k={k}"))
        out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30)), ye_arr, gmask,
                             f"[{tag}] DYN LCA k=30"))
        out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, 100, class_weighted=True), p=8),
                             ye_arr, gmask, f"[{tag}] DYN OLA-cw k=100 p=8"))
        for k in (30, 100):
            for p in (1, 3):
                out.append(score10_g(fuse_dyn(P_g, comp_ola(dist, idx, k, class_weighted=True),
                                              p=p, Wclass=W_class),
                                     ye_arr, gmask, f"[{tag}] HYBRID W x OLA-cw k={k} p={p}"))
        out.append(score10_g(fuse_dyn(P_g, comp_lca(P_g, idx, 30), p=1, Wclass=W_class),
                             ye_arr, gmask, f"[{tag}] HYBRID W x LCA k=30 p=1"))
    return pd.DataFrame(out)

spaces_selv = {"FEAT": (dF_selv, iF_selv), "DEC": (dD_selv, iD_selv)}
selv_tab = run_all(P_selv, spaces_selv, y_arr[selv_idx], gmask_selv).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print(selv_tab.to_string(index=False))

BESTV = selv_tab.iloc[0]["variant"]
b_feat = selv_tab[selv_tab.variant.str.startswith("[FEAT]")].iloc[0]
b_dec  = selv_tab[selv_tab.variant.str.startswith("[DEC]")].iloc[0]
v3row  = selv_tab[selv_tab.variant == "STATIC v3: per-class W"].iloc[0]
print("\nbest overall : %-38s %.4f" % (BESTV, selv_tab.iloc[0]["e2e_mF1"]))
print("best FEAT    : %-38s %.4f" % (b_feat["variant"], b_feat["e2e_mF1"]))
print("best DEC     : %-38s %.4f" % (b_dec["variant"],  b_dec["e2e_mF1"]))
print("STATIC v3    : %-38s %.4f" % ("per-class W", v3row["e2e_mF1"]))
print("\nDEC vs FEAT  : %+.4f    DEC vs v3 : %+.4f   (noise band +/-0.012)"
      % (b_dec["e2e_mF1"] - b_feat["e2e_mF1"], b_dec["e2e_mF1"] - v3row["e2e_mF1"]))

                           variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
        [DEC] DYN OLA-cw k=100 p=8   0.6317  0.6660  0.6337   0.9092
    [FEAT] HYBRID W x LCA k=30 p=1   0.6299  0.6825  0.6375   0.9081
 [DEC] HYBRID W x OLA-cw k=100 p=3   0.6295  0.6787  0.6401   0.9086
  [DEC] HYBRID W x OLA-cw k=30 p=1   0.6285  0.6740  0.6414   0.9085
 [FEAT] HYBRID W x OLA-cw k=30 p=3   0.6284  0.6758  0.6403   0.9082
               [FEAT] DYN LCA k=30   0.6279  0.6852  0.6097   0.9090
[FEAT] HYBRID W x OLA-cw k=100 p=3   0.6273  0.6768  0.6383   0.9084
  [DEC] HYBRID W x OLA-cw k=30 p=3   0.6271  0.6725  0.6388   0.9085
[FEAT] HYBRID W x OLA-cw k=100 p=1   0.6269  0.6738  0.6391   0.9082
            STATIC v3: per-class W   0.6269  0.6737  0.6389   0.9081
 [DEC] HYBRID W x OLA-cw k=100 p=1   0.6268  0.6735  0.6392   0.9083
 [FEAT] HYBRID W x OLA-cw k=30 p=1   0.6261  0.6724  0.6385   0.9082
     [DEC] HYBRID W x LCA k=30 p=1   0.6246  0.6684  0.6377   0.9079
       [FEAT] DYN OLA-cw k=100 p=8

## 6. Refit and evaluate on test once

In [8]:
t0 = time.time()
gate_f = fit_gate(X_trf, y_arr)
Xa_f   = X_trf[att_trf].reset_index(drop=True)
ya_f   = pd.Series(fam_trf[att_trf])
FINAL  = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
print("pool + gate refit in %.0fs" % (time.time() - t0))

gate_test  = gate_f.predict(X_test)
y_test_arr = y_test.values
gmask_test = gate_test == 1
gpos_test  = np.where(gmask_test)[0]
P_test     = [m.predict_proba(X_test.iloc[gpos_test]).astype(np.float32) for m in FINAL]

dF_test, iF_test = NN_F.kneighbors(scaler_F.transform(X_test.iloc[gpos_test]))
dD_test, iD_test = NN_D.kneighbors(profile(P_test))
print("neighbourhoods for %d gate-positive test rows" % len(gpos_test))

spaces_test = {"FEAT": (dF_test, iF_test), "DEC": (dD_test, iD_test)}
test_tab = run_all(P_test, spaces_test, y_test_arr, gmask_test).sort_values(
    "e2e_mF1", ascending=False).reset_index(drop=True)
print("\n--- TEST (selection already made on SELV) ---")
print(test_tab.to_string(index=False))

sel = test_tab[test_tab.variant == BESTV].iloc[0]
print("\nSELV-selected '%s' ranks %d of %d on test, e2e_mF1 %.4f, acc %.4f"
      % (BESTV, 1 + list(test_tab.variant).index(BESTV), len(test_tab),
         sel["e2e_mF1"], sel["e2e_acc"]))
print("vs v3 static (0.6334 here / 0.6339 in v3): %+.4f" % (sel["e2e_mF1"] - 0.6334))
print("vs v4-hybrid (0.6295): %+.4f" % (sel["e2e_mF1"] - 0.6295))
print("vs v4-dynamic (0.6262): %+.4f" % (sel["e2e_mF1"] - 0.6262))

pool + gate refit in 60s


neighbourhoods for 17633 gate-positive test rows



--- TEST (selection already made on SELV) ---
                           variant  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
        [DEC] DYN OLA-cw k=100 p=8   0.6363  0.6733  0.6380   0.9089
            STATIC v3: per-class W   0.6334  0.6821  0.6484   0.9086
[FEAT] HYBRID W x OLA-cw k=100 p=1   0.6334  0.6846  0.6476   0.9085
 [DEC] HYBRID W x OLA-cw k=100 p=1   0.6331  0.6843  0.6465   0.9088
[FEAT] HYBRID W x OLA-cw k=100 p=3   0.6331  0.6863  0.6458   0.9086
     [DEC] HYBRID W x LCA k=30 p=1   0.6329  0.6823  0.6463   0.9082
 [FEAT] HYBRID W x OLA-cw k=30 p=1   0.6327  0.6838  0.6471   0.9085
  [DEC] HYBRID W x OLA-cw k=30 p=3   0.6319  0.6850  0.6425   0.9084
 [FEAT] HYBRID W x OLA-cw k=30 p=3   0.6313  0.6834  0.6441   0.9084
  [DEC] HYBRID W x OLA-cw k=30 p=1   0.6308  0.6811  0.6446   0.9085
 [DEC] HYBRID W x OLA-cw k=100 p=3   0.6299  0.6834  0.6414   0.9088
    [FEAT] HYBRID W x LCA k=30 p=1   0.6295  0.6875  0.6394   0.9082
               [FEAT] DYN LCA k=30   0.6262  0.6965  0.6

### 6.1 Per-class report for the SELV-selected variant

In [9]:
def build(tag_variant, P_g, spaces):
    t = tag_variant
    if t == "STATIC v3: per-class W":   return fuse_static(P_g, W_class)
    if t == "STATIC: equal average":    return fuse_static(P_g, np.ones((M, NC)) / M)
    sp = t[1:t.index("]")]
    dist, idx = spaces[sp]
    body = t[t.index("]") + 2:]
    if body.startswith("DYN LCA k=30"):        return fuse_dyn(P_g, comp_lca(P_g, idx, 30))
    if body.startswith("HYBRID W x LCA"):      return fuse_dyn(P_g, comp_lca(P_g, idx, 30), p=1, Wclass=W_class)
    k = 100 if "k=100" in body else 30
    p = 8 if "p=8" in body else (3 if "p=3" in body else 1)
    Wd = comp_ola(dist, idx, k, class_weighted=True)
    if body.startswith("HYBRID"):              return fuse_dyn(P_g, Wd, p=p, Wclass=W_class)
    return fuse_dyn(P_g, Wd, p=(8 if "p=8" in body else 1))

fam_pred = build(BESTV, P_test, spaces_test)
yp = np.zeros(len(y_test_arr), int)
yp[gmask_test] = np.vectorize(idx_to_fam.get, otypes=[int])(fam_pred)
print(classification_report(y_test_arr, yp, labels=list(range(10)),
                            target_names=NAMES, digits=3, zero_division=0))

v4h = dict(zip(NAMES, [0.983, 0.453, 0.662, 0.449, 0.784, 0.711, 0.814, 0.643, 0.358, 0.438]))
v5  = f1_score(y_test_arr, yp, average=None, labels=range(10), zero_division=0)
print(pd.DataFrame({"v4-hybrid": [v4h[n] for n in NAMES], "v5": v5.round(3),
                    "delta": (v5 - np.array([v4h[n] for n in NAMES])).round(3)},
                   index=NAMES).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.968     0.983     45001
      Analysis      0.346     0.701     0.464        77
      Backdoor      0.750     0.533     0.623        90
           DoS      0.654     0.332     0.440       883
      Exploits      0.818     0.757     0.786      6182
       Fuzzers      0.587     0.905     0.712      5279
       Generic      0.829     0.806     0.817       899
Reconnaissance      0.745     0.573     0.648      2406
     Shellcode      0.387     0.368     0.377       370
         Worms      0.618     0.438     0.512        48

      accuracy                          0.909     61235
     macro avg      0.673     0.638     0.636     61235
  weighted avg      0.922     0.909     0.911     61235

                v4-hybrid     v5  delta
Benign              0.983  0.983 -0.000
Analysis            0.453  0.464  0.011
Backdoor            0.662  0.623 -0.039
DoS                 0.449  0.440 -0.009
Exploits     

## 7. Findings

**v5 = test macro-F1 0.6363, accuracy 0.9089** - the best score recorded in this project, SELV-selected
and **ranked 1st of 20 on test**. The winner is `[DEC] DYN OLA-cw k=100 p=8`: purely dynamic, no
per-class matrix, competence estimated in **decision space**.

| Model | Space | Combiner | **Test macro-F1** | **Test acc** |
|---|---|---|---|---|
| **v5** `DYN OLA-cw k=100 p=8` | **DEC** | pure dynamic | **0.6363** | **0.9089** |
| v3 static per-class $W$ | - | static | 0.6334 | 0.9086 |
| v4-hybrid `HYBRID W x LCA k=30 p=1` | FEAT | hybrid | 0.6295 | 0.9082 |
| v4-dynamic `DYN LCA k=30` | FEAT | pure dynamic | 0.6262 | 0.9085 |

**+0.0029 over v3 and +0.0068 over v4-hybrid - both inside the +/-0.012 paired band.** Report v5 as
`0.636 [0.618, 0.653]` and state the v3 comparison as *within noise*. What is genuinely new is not the
margin but that a **purely dynamic** combiner reached the top of the table at all, and that the SELV
choice transferred (v4's did not).

### 1. Decision space beats feature space on every measure

| | FEAT | DEC |
|---|---|---|
| Neighbourhood purity k=10 | 0.5721 | **0.6698** |
| k=30 | 0.5465 | **0.6686** |
| k=100 | 0.5147 | **0.6666** |
| Mean per-member competence AUC | 0.8193 | **0.8539** |
| Best variant (SELV) | 0.6299 | **0.6317** |
| Best variant (test) | 0.6334 | **0.6363** |

Feature-space purity **degrades** as the neighbourhood widens (0.572 -> 0.515) while decision-space
purity is **flat** (0.670 -> 0.667). That is the geometric signature of the k-NN overlap result:
widening a feature-space ball pulls in other families, whereas the pool's output profile stays
coherent. **v4 used the non-standard space; DES formulations that define competence on output
profiles (KNORA, META-DES) are right to do so.**

### 2. CORRECTION TO v4: competence is NOT unestimable

v4 concluded *"local-accuracy competence cannot be estimated finely enough"*. **That conclusion is
wrong and this notebook retracts it.** Per-member AUC for predicting actual correctness is
**0.82-0.85 in both spaces** - nowhere near the 0.50 that would indicate noise. The estimator works.

The real mechanism, measured by decomposing $w_m(x)$ into a shared component and a member-specific
residual over 35,153 gate-positive SELV rows:

| Quantity | Value |
|---|---|
| Shared variance (row difficulty) | **98.1%** |
| Member-specific variance | **1.9%** |
| Mean pairwise correlation of $w_m$ across members | **0.9818** |
| Median within-row spread $\max_m w - \min_m w$ | **0.0000** |
| Competence-argmax picks a correct member | 0.7411 |
| Always using the single best member | 0.7417 |

**The competence estimate is a row-difficulty signal, not a member-discrimination signal.** All seven
members' competences move together ($r = 0.98$); on the median row they are *identical*. Each $w_m$
therefore predicts its own member's correctness well (high AUC) while the *relative ordering* across
members carries almost nothing - which is precisely why the weights sit at uniform (mean |dev| 0.0060)
and why selecting by competence (0.7411) cannot beat always using the strongest member (0.7417).

This supersedes the v4 framing. **State it as: competence is estimable, but what is estimable is how
hard the row is, not who should handle it.** It explains DES v1, v4 and v5 with one measurement, and
it predicts the failure of any scheme whose weights derive from local accuracy - including KNORA and
META-DES variants - without needing to run them.

It also explains the winning hyper-parameter: **p=8**. Heavy sharpening is required to amplify a
residual that is only 1.9% of the variance. The gain is real but it is being squeezed from a very thin
signal, which is consistent with its size (+0.0029, inside noise).

### 3. Why the pure-dynamic form won here

In FEAT space the hybrid always beat pure dynamic, because the per-class matrix carried the
information the competence term lacked. In DEC space the ordering flips at the top: `DYN OLA-cw k=100
p=8` (0.6363) edges the best DEC hybrid (0.6331). With purer neighbourhoods the sharpened competence
term finally contributes something the per-class prior does not - though note every hybrid variant in
both spaces lands in a tight 0.6299-0.6334 band, so this ordering is itself inside noise.

### 4. Per-class movement (v5 vs v4-hybrid)

| Family | v4-hybrid | v5 | delta |
|---|---|---|---|
| **Worms** | 0.438 | 0.512 | **+0.074** |
| Shellcode | 0.358 | 0.377 | +0.019 |
| Analysis | 0.453 | 0.464 | +0.011 |
| Reconnaissance | 0.643 | 0.648 | +0.005 |
| Generic | 0.814 | 0.817 | +0.003 |
| Exploits | 0.784 | 0.786 | +0.002 |
| Fuzzers | 0.711 | 0.712 | +0.001 |
| Benign | 0.983 | 0.983 | -0.000 |
| DoS | 0.449 | 0.440 | -0.009 |
| **Backdoor** | 0.662 | 0.623 | **-0.039** |

Worms (48 rows, +/-0.13) and Backdoor (90 rows) dominate the deltas and neither is established. The
five largest families all move by <= 0.005. **The macro-F1 gain is concentrated in exactly the classes
whose intervals cannot support it** - another reason to treat +0.0029 as within noise rather than as a
result.

### Recommendation

**Adopt `[DEC] DYN OLA-cw k=100 p=8` as the reported dynamic model (0.636 [0.618, 0.653],
accuracy 0.909)**, keeping v3 static as the matched baseline and v4 as the feature-space ablation. The
three together make a complete argument: same pool, same protocol, three combiners, and a measured
explanation of why they land within 0.01 of each other.

**Do not pursue further competence estimators.** The 98.1%/1.9% decomposition bounds the whole family:
any weighting derived from local accuracy inherits a signal that is 98% row difficulty. Learned
competence (META-DES style) would be fitting the same quantity with a better function approximator and
is predicted to fail for the same reason. **This closes option B from the improvement plan without
needing to run it.**

The remaining levers are unchanged and lie outside the combiner: **gate precision** (option C - the
gate still uses `balanced` weighting, a lesson never carried over from Stage B) and **derived
features** (option D - the only lever that moves the Cover-Hart bound rather than approaching it).